In [0]:
import os
import sys

src_path = os.path.abspath("../src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

from config import LANDING, BRONZE_TABLE
from bronze import (
    create_bronze_table,
    load_bronze,
    rows_per_source_file,
    reconcile,
    check_idempotency
)

print("Landing path:", LANDING)
print("Bronze table:", BRONZE_TABLE)

In [0]:
landing_files = sorted(f for f in os.listdir(LANDING) if f.endswith(".csv"))

if not landing_files:
    raise ValueError(
        f"No CSV files found in landing: {LANDING}. "
        "Run 02_seed_landing first"
    )

print("Number of CSV files:", len(landing_files))

for filename in landing_files:
    print(filename)

In [0]:
create_bronze_table(spark)

print("Bronze table is ready:", BRONZE_TABLE)

In [0]:
load_bronze(spark)

print("Bronze ingestion completed.")

In [0]:
bronze_df = spark.table(BRONZE_TABLE)

print("Bronze schema:")
bronze_df.printSchema()

print("Sample records:")
display(bronze_df.limit(10))

In [0]:
file_counts_df = rows_per_source_file(spark)

display(file_counts_df)

In [0]:
reconciliation_passed = reconcile(spark)

if not reconciliation_passed:
    raise ValueError("Bronze reconciliation failed.")

idempotency_passed = check_idempotency(spark)

if not idempotency_passed:
    raise ValueError("Bronze idempotency check failed.")